In [2]:
# from transformers import BertModel, BertConfig
from transformers import AutoTokenizer
from datasets import load_dataset
from ray.tune.search.hyperopt import HyperOptSearch
from ray.tune.schedulers import ASHAScheduler


In [3]:
dataset = load_dataset("yelp_review_full")

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
import transformers
transformers.__version__

'4.37.1'

In [5]:
dataset["train"][100]

{'label': 0,
 'text': 'My expectations for McDonalds are t rarely high. But for one to still fail so spectacularly...that takes something special!\\nThe cashier took my friends\'s order, then promptly ignored me. I had to force myself in front of a cashier who opened his register to wait on the person BEHIND me. I waited over five minutes for a gigantic order that included precisely one kid\'s meal. After watching two people who ordered after me be handed their food, I asked where mine was. The manager started yelling at the cashiers for \\"serving off their orders\\" when they didn\'t have their food. But neither cashier was anywhere near those controls, and the manager was the one serving food to customers and clearing the boards.\\nThe manager was rude when giving me my order. She didn\'t make sure that I had everything ON MY RECEIPT, and never even had the decency to apologize that I felt I was getting poor service.\\nI\'ve eaten at various McDonalds restaurants for over 30 years. 

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):

    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/650000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [8]:
type(tokenized_datasets)

datasets.dataset_dict.DatasetDict

In [9]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(50))

small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(50))

In [10]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/home/k2141255/.local/lib/python3.8/site-packages/torch/cuda/__init__.py:611: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid us

In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(output_dir="test_trainer", evaluation_strategy="epoch")

In [12]:
import numpy as np

from datasets import load_metric

metric = load_metric("accuracy")

/tmp/ipykernel_3508/356211072.py:5: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric("accuracy")
/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/datasets/load.py:752: FutureWarning: The repository for accuracy contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.16.1/metrics/accuracy/accuracy.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


In [13]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return metric.compute(predictions=predictions, references=labels)

In [14]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

In [15]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir="test_trainer", evaluation_strategy="epoch")

In [16]:
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', return_dict=True, force_download=True)

In [20]:
 trainer = Trainer(

#     model=model,

    args=training_args,

    train_dataset=small_train_dataset,

    eval_dataset=small_eval_dataset,

    compute_metrics=compute_metrics,
    
    model_init=model_init

    )

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
best_trial = trainer.hyperparameter_search(
    direction="maximize",
    backend="ray",
    # Choose among many libraries:
    # https://docs.ray.io/en/latest/tune/api_docs/suggestion.html
    search_alg=HyperOptSearch(metric="objective", mode="max"),
    # Choose among schedulers:
    # https://docs.ray.io/en/latest/tune/api_docs/schedulers.html
    scheduler=ASHAScheduler(metric="objective", mode="max"))


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

== Status ==
Current time: 2024-08-15 18:08:33 (running for 00:00:00.11)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 16.000: None | Iter 4.000: None | Iter 1.000: None
Logical resource usage: 0/16 CPUs, 0/0 GPUs
Result logdir: /home/k2141255/ray_results/_objective_2024-08-15_18-08-33
Number of trials: 1/20 (1 PENDING)
+---------------------+----------+-------+-----------------+--------------------+------------------------+---------+
| Trial name          | status   | loc   |   learning_rate |   num_train_epochs |   per_device_train_bat |    seed |
|                     |          |       |                 |                    |                ch_size |         |
|---------------------+----------+-------+-----------------+--------------------+------------------------+---------|
| _objective_b902eaed | PENDING  |       |     2.40538e-06 |                  1 |                      8 | 39.8733 |
+---------------------+----------+-------+-----------------+---------

config.json: 100%|██████████| 483/483 [00:00<00:00, 228kB/s]
model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]
model.safetensors:   4%|▍         | 10.5M/268M [00:00<00:03, 64.5MB/s]
model.safetensors:  12%|█▏        | 31.5M/268M [00:00<00:02, 91.0MB/s]
model.safetensors:  16%|█▌        | 41.9M/268M [00:00<00:02, 95.3MB/s]
model.safetensors:  20%|█▉        | 52.4M/268M [00:00<00:02, 93.8MB/s]
model.safetensors:  27%|██▋       | 73.4M/268M [00:00<00:01, 104MB/s] 
model.safetensors:  31%|███▏      | 83.9M/268M [00:00<00:01, 103MB/s]
model.safetensors:  39%|███▉      | 105M/268M [00:01<00:01, 105MB/s] 
model.safetensors:  47%|████▋     | 126M/268M [00:01<00:01, 105MB/s]
model.safetensors:  55%|█████▍    | 147M/268M [00:01<00:01, 106MB/s]
model.safetensors:  63%|██████▎   | 168M/268M [00:01<00:00, 105MB/s]


== Status ==
Current time: 2024-08-15 18:08:38 (running for 00:00:05.13)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 16.000: None | Iter 4.000: None | Iter 1.000: None
Logical resource usage: 3.0/16 CPUs, 0/0 GPUs
Result logdir: /home/k2141255/ray_results/_objective_2024-08-15_18-08-33
Number of trials: 3/20 (1 PENDING, 2 RUNNING)
+---------------------+----------+--------------------+-----------------+--------------------+------------------------+----------+
| Trial name          | status   | loc                |   learning_rate |   num_train_epochs |   per_device_train_bat |     seed |
|                     |          |                    |                 |                    |                ch_size |          |
|---------------------+----------+--------------------+-----------------+--------------------+------------------------+----------|
| _objective_b902eaed | RUNNING  | 10.151.122.12:4592 |     2.40538e-06 |                  1 |                      8

model.safetensors:  70%|███████   | 189M/268M [00:01<00:00, 107MB/s]
model.safetensors:  78%|███████▊  | 210M/268M [00:02<00:00, 109MB/s]
model.safetensors:  86%|████████▌ | 231M/268M [00:02<00:00, 108MB/s]
model.safetensors:  94%|█████████▍| 252M/268M [00:02<00:00, 109MB/s]
model.safetensors: 100%|██████████| 268M/268M [00:02<00:00, 105MB/s]
(_objective pid=4592) Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
(_objective pid=4592) You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
config.json: 100%|██████████| 483/483 [00:00<00:00, 214kB/s] [repeated 4x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/ray-logging.html#log-dedupl

...

== Status ==
Current time: 2024-08-15 18:09:19 (running for 00:00:45.54)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 64.000: None | Iter 16.000: None | Iter 4.000: None | Iter 1.000: None
Logical resource usage: 10.0/16 CPUs, 0/0 GPUs
Result logdir: /home/k2141255/ray_results/_objective_2024-08-15_18-08-33
Number of trials: 17/20 (7 ERROR, 1 PENDING, 9 RUNNING)
+---------------------+----------+--------------------+-----------------+--------------------+------------------------+----------+
| Trial name          | status   | loc                |   learning_rate |   num_train_epochs |   per_device_train_bat |     seed |
|                     |          |                    |                 |                    |                ch_size |          |
|---------------------+----------+--------------------+-----------------+--------------------+------------------------+----------|
| _objective_d36f3f45 | RUNNING  | 10.151.122.12:5010 |     2.87755e-05 |                  5 |            

(_objective pid=5666) Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
(_objective pid=5666) You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


(raylet) A worker died or was killed while executing a task by an unexpected system error. To troubleshoot the problem, check the logs for the dead worker. RayTask ID: ffffffffffffffff60f1fd1dc1b8d9553d02e14e01000000 Worker ID: 7ae9a315c2e03c0027acc5f7deb8b471e2b1c4274e7c4f2adf579edd Node ID: 120676c54555e4d689ce4fa5d243d108ef05481ae602d09ba7df3677 Worker IP address: 10.151.122.12 Worker port: 39707 Worker PID: 5010 Worker exit type: SYSTEM_ERROR Worker exit detail: Worker unexpectedly exits with a connection error code 2. End of file. There are some potential root causes. (1) The process is killed by SIGKILL by OOM killer due to high memory usage. (2) ray stop --force is called. (3) The worker is crashed unexpectedly due to SIGSEGV or other unexpected errors.


model.safetensors: 100%|██████████| 268M/268M [00:02<00:00, 90.8MB/s] [repeated 3x across cluster]
model.safetensors:  90%|█████████ | 241M/268M [00:02<00:00, 101MB/s] [repeated 9x across cluster]
  0%|          | 0/4 [00:18<?, ?it/s]
(_objective pid=5156) [2024-08-15 18:09:30,557 E 5156 5226] logging.cc:104: Stack trace: 
(_objective pid=5156)  /home/k2141255/.local/lib/python3.8/site-packages/ray/_raylet.so(+0x10051da) [0x7fa307d5b1da] ray::operator<<()
(_objective pid=5156) /home/k2141255/.local/lib/python3.8/site-packages/ray/_raylet.so(+0x1007918) [0x7fa307d5d918] ray::TerminateHandler()
(_objective pid=5156) /home/k2141255/anaconda3/envs/study_design_classifier/bin/../lib/libstdc++.so.6(+0xb135a) [0x7fa306bcb35a] __cxxabiv1::__terminate()
(_objective pid=5156) /home/k2141255/anaconda3/envs/study_design_classifier/bin/../lib/libstdc++.so.6(+0xb13c5) [0x7fa306bcb3c5]
(_objective pid=5156) /home/k2141255/anaconda3/envs/study_design_classifier/bin/../lib/libstdc++.so.6(__gxx_personal

RuntimeError: Caught unexpected exception: Task was killed due to the node running low on memory.
Memory on the node (IP: 10.151.122.12, ID: 120676c54555e4d689ce4fa5d243d108ef05481ae602d09ba7df3677) where the task (actor ID: 366b8fad5b6ab5167249551f01000000, name=ImplicitFunc.__init__, pid=5666, memory used=1.04GB) was running was 60.16GB / 62.62GB (0.960712), which exceeds the memory usage threshold of 0.95. Ray killed this worker (ID: d9d5367a36978699723cc6e3ae44288bbc7b11c225857035e5a28e4a) because it was the most recently scheduled task; to see more information about memory usage on this node, use `ray logs raylet.out -ip 10.151.122.12`. To see the logs of the worker, use `ray logs worker-d9d5367a36978699723cc6e3ae44288bbc7b11c225857035e5a28e4a*out -ip 10.151.122.12. Top 10 memory users:
PID	MEM(GB)	COMMAND
5010	12.04	ray::ImplicitFunc.train
5156	9.78	ray::ImplicitFunc.train
5300	9.06	ray::ImplicitFunc.train
5227	6.43	ray::ImplicitFunc.train
5372	6.15	ray::ImplicitFunc.train
5521	4

In [20]:
small_train_dataset['label']

Downloading:  68%|██████▊   | 175M/256M [00:20<00:23, 3.56MB/s]


[4,
 2,
 4,
 0,
 2,
 2,
 3,
 0,
 0,
 4,
 0,
 4,
 3,
 1,
 3,
 0,
 4,
 3,
 4,
 2,
 0,
 4,
 0,
 1,
 0,
 1,
 1,
 2,
 4,
 3,
 3,
 2,
 4,
 3,
 0,
 2,
 2,
 0,
 1,
 2,
 2,
 2,
 0,
 1,
 4,
 1,
 1,
 2,
 0,
 2]

Downloading:  68%|██████▊   | 175M/256M [00:20<00:24, 3.51MB/s]
Downloading:  69%|██████▊   | 175M/256M [00:20<00:24, 3.41MB/s]
Downloading:  69%|██████▊   | 176M/256M [00:20<00:26, 3.17MB/s]
Downloading:  69%|██████▉   | 176M/256M [00:20<00:25, 3.32MB/s]
Downloading:  69%|██████▉   | 176M/256M [00:20<00:24, 3.34MB/s]
Downloading:  69%|██████▉   | 177M/256M [00:21<00:22, 3.64MB/s]
Downloading:  69%|██████▉   | 177M/256M [00:21<00:21, 3.88MB/s]
Downloading:  69%|██████▉   | 178M/256M [00:21<00:20, 4.05MB/s]
Downloading:  70%|██████▉   | 178M/256M [00:21<00:19, 4.16MB/s]
Downloading:  70%|██████▉   | 178M/256M [00:21<00:18, 4.29MB/s]
Downloading:  70%|██████▉   | 179M/256M [00:21<00:18, 4.31MB/s]
Downloading:  70%|███████   | 179M/256M [00:21<00:18, 4.39MB/s]
Downloading:  70%|███████   | 180M/256M [00:21<00:17, 4.47MB/s]
Downloading:  71%|███████   | 180M/256M [00:21<00:16, 4.66MB/s]
Downloading:  71%|███████   | 181M/256M [00:21<00:15, 4.98MB/s]
Downloading:  71%|███████   | 181M/256M 

Downloading:  95%|█████████▍| 242M/256M [00:35<00:05, 2.44MB/s]
Downloading:  95%|█████████▍| 242M/256M [00:35<00:05, 2.44MB/s]
Downloading:  95%|█████████▍| 243M/256M [00:35<00:06, 1.95MB/s]
Downloading:  95%|█████████▌| 243M/256M [00:35<00:08, 1.61MB/s]
Downloading:  95%|█████████▌| 243M/256M [00:36<00:07, 1.85MB/s]
Downloading:  95%|█████████▌| 243M/256M [00:36<00:06, 1.89MB/s]
Downloading:  95%|█████████▌| 244M/256M [00:36<00:06, 1.92MB/s]
Downloading:  95%|█████████▌| 244M/256M [00:36<00:05, 2.19MB/s]
Downloading:  96%|█████████▌| 244M/256M [00:36<00:05, 2.38MB/s]
Downloading:  96%|█████████▌| 244M/256M [00:36<00:05, 2.31MB/s]
Downloading:  96%|█████████▌| 245M/256M [00:36<00:05, 2.17MB/s]
Downloading:  96%|█████████▌| 245M/256M [00:36<00:05, 2.06MB/s]
Downloading:  96%|█████████▌| 245M/256M [00:36<00:04, 2.27MB/s]
Downloading:  96%|█████████▌| 245M/256M [00:37<00:04, 2.37MB/s]
Downloading:  96%|█████████▌| 246M/256M [00:37<00:03, 2.71MB/s]
Downloading:  96%|█████████▋| 246M/256M 

Downloading:  10%|▉         | 24.5M/256M [00:04<01:01, 3.94MB/s]
Downloading:  10%|▉         | 25.0M/256M [00:04<00:56, 4.28MB/s]
Downloading:  10%|▉         | 25.4M/256M [00:04<01:07, 3.58MB/s]
Downloading:  10%|█         | 26.3M/256M [00:04<00:48, 4.99MB/s]
Downloading:  11%|█         | 27.2M/256M [00:04<00:39, 6.05MB/s]
Downloading:  11%|█         | 28.2M/256M [00:05<00:33, 7.14MB/s]
Downloading:  11%|█▏        | 29.3M/256M [00:05<00:28, 8.27MB/s]
Downloading:  12%|█▏        | 30.2M/256M [00:05<00:27, 8.60MB/s]
Downloading:  12%|█▏        | 31.1M/256M [00:05<00:26, 8.90MB/s]
Downloading:  13%|█▎        | 32.2M/256M [00:05<00:24, 9.55MB/s]
Downloading:  13%|█▎        | 33.3M/256M [00:05<00:23, 10.0MB/s]
Downloading:  13%|█▎        | 34.3M/256M [00:05<00:24, 9.49MB/s]
Downloading:  14%|█▍        | 35.2M/256M [00:05<00:25, 9.23MB/s]
Downloading:  14%|█▍        | 36.1M/256M [00:05<00:24, 9.23MB/s]
Downloading:  14%|█▍        | 37.0M/256M [00:05<00:25, 9.16MB/s]
Downloading:  15%|█▍     

Downloading:  56%|█████▌    | 143M/256M [00:16<00:05, 20.9MB/s]
Downloading:  57%|█████▋    | 147M/256M [00:16<00:04, 24.5MB/s]
Downloading:  59%|█████▊    | 150M/256M [00:16<00:04, 27.6MB/s]
Downloading:  60%|██████    | 154M/256M [00:16<00:03, 30.5MB/s]
Downloading:  62%|██████▏   | 158M/256M [00:16<00:03, 32.8MB/s]
Downloading:  63%|██████▎   | 161M/256M [00:16<00:02, 34.6MB/s]
Downloading:  65%|██████▍   | 165M/256M [00:17<00:02, 35.6MB/s]
Downloading:  66%|██████▌   | 168M/256M [00:17<00:02, 31.8MB/s]
Downloading:  68%|██████▊   | 174M/256M [00:17<00:02, 38.5MB/s]
Downloading:  70%|██████▉   | 178M/256M [00:17<00:02, 39.0MB/s]
Downloading:  71%|███████   | 182M/256M [00:17<00:02, 38.7MB/s]
Downloading:  73%|███████▎  | 185M/256M [00:17<00:01, 38.7MB/s]
Downloading:  74%|███████▍  | 189M/256M [00:17<00:01, 38.5MB/s]
Downloading:  75%|███████▌  | 193M/256M [00:17<00:01, 38.6MB/s]
Downloading:  77%|███████▋  | 197M/256M [00:17<00:01, 37.1MB/s]
Downloading:  78%|███████▊  | 200M/256M 

Downloading:  41%|████      | 105M/256M [00:04<00:10, 15.5MB/s]
Downloading:  42%|████▏     | 107M/256M [00:04<00:14, 11.0MB/s]
Downloading:  43%|████▎     | 110M/256M [00:04<00:11, 13.9MB/s]
Downloading:  44%|████▍     | 114M/256M [00:05<00:08, 18.1MB/s]
Downloading:  46%|████▌     | 118M/256M [00:05<00:06, 22.7MB/s]
Downloading:  47%|████▋     | 121M/256M [00:05<00:09, 14.2MB/s]
Downloading:  48%|████▊     | 123M/256M [00:06<00:14, 9.34MB/s]
Downloading:  49%|████▊     | 125M/256M [00:06<00:18, 7.35MB/s]
Downloading:  49%|████▉     | 126M/256M [00:06<00:17, 7.82MB/s]
Downloading:  50%|████▉     | 127M/256M [00:07<00:21, 6.36MB/s]
Downloading:  50%|█████     | 128M/256M [00:07<00:21, 6.34MB/s]
Downloading:  50%|█████     | 129M/256M [00:07<00:19, 6.65MB/s]
Downloading:  51%|█████     | 131M/256M [00:07<00:15, 8.66MB/s]
Downloading:  53%|█████▎    | 134M/256M [00:07<00:09, 13.9MB/s]
Downloading:  54%|█████▍    | 138M/256M [00:07<00:06, 19.3MB/s]
Downloading:  55%|█████▌    | 142M/256M 

(_objective pid=903367) 2022-08-01 10:35:40,985	ERROR function_runner.py:281 -- Runner Thread raised error.
(_objective pid=903367) Traceback (most recent call last):
(_objective pid=903367)   File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 272, in run
(_objective pid=903367)     self._entrypoint()
(_objective pid=903367)   File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 348, in entrypoint
(_objective pid=903367)     return self._trainable_func(
(_objective pid=903367)   File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/util/tracing/tracing_helper.py", line 462, in _resume_span
(_objective pid=903367)     return method(self, *_args, **_kwargs)
(_objective pid=903367)   File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 640, in

2022-08-01 10:35:46,898	ERROR worker.py:92 -- Unhandled error (suppress with 'RAY_IGNORE_UNHANDLED_ERRORS=1'): ray::ImplicitFunc.train() (pid=903367, ip=10.151.122.12, repr=_objective)
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/trainable.py", line 349, in train
    result = self.step()
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 403, in step
    self._report_thread_runner_error(block=True)
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 567, in _report_thread_runner_error
    raise TuneError(
ray.tune.error.TuneError: Trial raised an exception. Traceback:
ray::ImplicitFunc.train() (pid=903367, ip=10.151.122.12, repr=_objective)
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 272, in run
    self._ent